In [13]:
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import nltk

In [11]:
# Download NLTK punkt tokenizer
nltk.download('punkt')

# Load LLaMA tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading checkpoint shards: 100%|██████████| 4/4 [00:18<00:00,  4.61s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [ ]:
def generate_and_save_rdf(prompt, top_k=3, n_steps=5, output_file="branching_structure.ttl"):
    # Initialize an RDF graph and namespace
    g = Graph()
    EX = Namespace("http://example.org/")
    g.bind("ex", EX)

    # Tokenize the initial prompt
    tokens = tokenizer(prompt, return_tensors='pt')
    input_ids = tokens.input_ids

    # Track each token generated in each step
    current_nodes = [input_ids]  # Start with the initial prompt as the root node

    for step in range(n_steps):
        next_nodes = []  # Reset next_nodes for each step to only hold new branches

        for branch_id, branch_input_ids in enumerate(current_nodes):
            # Generate top-k next tokens for each current token/branch
            with torch.no_grad():
                outputs = model(branch_input_ids)
                logits = outputs.logits[:, -1, :]

                # Ensure exactly top_k next tokens are selected
                top_k_probs, top_k_indices = torch.topk(logits, top_k, dim=-1)

                # Iterate over each of the top-k options and create branches
                for k in range(top_k):
                    next_token = top_k_indices[0, k].item()
                    next_token_text = tokenizer.decode([next_token], skip_special_tokens=True)
                    new_input_ids = torch.cat([branch_input_ids, torch.tensor([[next_token]])], dim=-1)

                    # Create URIs for the current and next nodes
                    current_uri = EX[f"Branch_{step}_{branch_id}"]
                    next_uri = EX[f"Branch_{step+1}_{branch_id}_{k}"]

                    # Add current token's branch node if not already present
                    if (current_uri, RDF.type, EX.Token) not in g:
                        g.add((current_uri, RDF.type, EX.Token))
                        g.add((current_uri, EX.text, Literal(tokenizer.decode(branch_input_ids[0], skip_special_tokens=True))))
                        g.add((current_uri, EX.iteration, Literal(step)))

                    # Add next token node
                    g.add((next_uri, RDF.type, EX.Token))
                    g.add((next_uri, EX.text, Literal(next_token_text)))
                    g.add((next_uri, EX.iteration, Literal(step + 1)))
                    # Link the current token to the next token
                    g.add((current_uri, EX.next, next_uri))

                    # Save this new token sequence for the next step
                    next_nodes.append(new_input_ids)

        # Update current_nodes for the next iteration to only include exactly top_k branches
        current_nodes = next_nodes

    # Save the entire RDF graph to a single Turtle file
    g.serialize(destination=output_file, format="turtle")
    print(f"Saved branching structure to RDF file: {output_file}")


In [ ]:
# Example usage
prompt = "Once upon a time in a distant land"
output = generate_and_save_rdf(prompt, top_k=3, n_steps=3)

Saved branching structure to RDF file: branching_structure.ttl


# Test 2

In [26]:
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import nltk

nltk.download('punkt')

# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.48s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [31]:
from rdflib import Graph, Literal, RDF, Namespace
import torch

# Function to generate RDF with full paths from root, coherence scores, and token representations
def generate_and_save_rdf_with_classes(prompt, top_k=3, n_steps=5, output_file="branching_structure_with_classes.ttl"):
    # Initialize an RDF graph
    g = Graph()
    EX = Namespace("http://example.org/")
    g.bind("ex", EX)

    # Tokenize the initial prompt
    tokens = tokenizer(prompt, return_tensors='pt')
    input_ids = tokens.input_ids

    # Define the root node with the original prompt text
    root_node = EX["Root"]
    g.add((root_node, RDF.type, EX.Path))
    g.add((root_node, EX.text, Literal(prompt)))

    # Track each token generated at each step as branches from the root node
    for step in range(n_steps):
        print(f"\nStep {step + 1}/{n_steps}", end=" ")

        # Generate top-k branches for each step
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits[:, -1, :]

            # Get top-k tokens for branching
            top_k_probs, top_k_indices = torch.topk(logits, top_k, dim=-1)

            for k in range(top_k):
                print(f"Token {k + 1}/{top_k}", end=" ")
                next_token = top_k_indices[0, k].item()
                next_token_text = tokenizer.decode([next_token], skip_special_tokens=True).strip()

                # Full path text from root to this token
                full_path_text = prompt + " " + next_token_text

                # Generate coherence scores for this path from root to token
                coherence_acc_long, coherence_acc_short, coherence_diff = coherence_metrics(full_path_text)

                # URI for the full path and token nodes
                full_path_uri = EX[f"Path_{step+1}_{k}"]
                next_token_uri = EX[f"Token_{step+1}_{k}"]

                # Add RDF triples for the full path from root to this token
                g.add((full_path_uri, RDF.type, EX.Path))
                g.add((full_path_uri, EX.text, Literal(full_path_text)))
                g.add((full_path_uri, EX.coherenceScore, Literal(coherence_acc_short)))
                g.add((full_path_uri, EX.coherenceDifference, Literal(coherence_diff)))
                g.add((root_node, EX.next, full_path_uri))

                # Individual token representation with coherence info
                g.add((next_token_uri, RDF.type, EX.Token))
                g.add((next_token_uri, EX.text, Literal(next_token_text)))
                g.add((next_token_uri, EX.iteration, Literal(step + 1)))
                g.add((full_path_uri, EX.next, next_token_uri))

    # Save the entire RDF graph to a Turtle file
    g.serialize(destination=output_file, format="turtle")
    print(f"Saved branching structure with classes to RDF file: {output_file}")


In [32]:
# Example usage
prompt = "Once upon a time"
generate_and_save_rdf_with_classes(prompt, top_k=3, n_steps=4, output_file="branching_structure_with_classes.ttl")


Step 1/4 Token 1/3 Token 2/3 Token 3/3 
Step 2/4 Token 1/3 Token 2/3 Token 3/3 
Step 3/4 Token 1/3 Token 2/3 Token 3/3 
Step 4/4 Token 1/3 Token 2/3 Token 3/3 Saved branching structure with classes to RDF file: branching_structure_with_classes.ttl
